# Imports


In [7]:
# Import all libraries used throughout the notebook.

import os
import random
import xml.etree.ElementTree as ET

import albumentations as A
import cv2
import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset


# Reproducibility


In [8]:
# Set fixed random seeds for reproducible splits, sampling, and model initialization.

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print(f"Seed set to {SEED}")


Seed set to 42


# Exploratory Data Analysis (Already in a separate notebook)


In [9]:
# Define the dataset configuration and preview the folder structure.

DATASET_PATH = "PCB-DATASET-master"

CLASSES = [
    "Missing_hole",
    "Mouse_bite",
    "Open_circuit",
    "Short",
    "Spur",
    "Spurious_copper"
]

for root, dirs, files in os.walk(DATASET_PATH):
    level = root.replace(DATASET_PATH, '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    
    if level < 2:  # prevent huge output
        subindent = ' ' * 4 * (level + 1)
        for f in files[:10]:
            print(f"{subindent}{f}")


PCB-DATASET-master/
    README.md
    rotate.py
    Annotations/
        Missing_hole/
        Mouse_bite/
        Open_circuit/
        Short/
        Spur/
        Spurious_copper/
    images/
        Missing_hole/
        Mouse_bite/
        Open_circuit/
        Short/
        Spur/
        Spurious_copper/
    PCB_USED/
        01.JPG
        04.JPG
        05.JPG
        06.JPG
        07.JPG
        08.JPG
        09.JPG
        10.JPG
        11.JPG
        12.JPG
    rotation/
        Missing_hole_angles.txt
        Mouse_bite_angles.txt
        Open_circuit_angles.txt
        Short_angles.txt
        Spurious_copper_angles.txt
        Spur_angles.txt
        Missing_hole_rotation/
        Mouse_bite_rotation/
        Open_circuit_rotation/
        Short_rotation/
        Spurious_copper_rotation/
        Spur_rotation/


# Model: From Scratch YOLO-Style PCB Defect Detector (Failed Trial)


## Phase 1: Build Annotations, Split the Dataset, and Prepare DataLoaders


In [10]:
# Build class-to-index and index-to-class lookup dictionaries.

CLASS_TO_ID = {
    cls: idx
    for idx, cls in enumerate(CLASSES)
}

print(CLASS_TO_ID)

{'Missing_hole': 0, 'Mouse_bite': 1, 'Open_circuit': 2, 'Short': 3, 'Spur': 4, 'Spurious_copper': 5}


In [11]:
# Parse XML annotation files into a DataFrame of image paths, classes, and bounding boxes.

records = []

for cls in CLASSES:

    img_dir = os.path.join(
        DATASET_PATH,
        "images",
        cls
    )

    ann_dir = os.path.join(
        DATASET_PATH,
        "Annotations",
        cls
    )

    for xml_file in os.listdir(ann_dir):

        if not xml_file.endswith(".xml"):
            continue

        xml_path = os.path.join(
            ann_dir,
            xml_file
        )

        tree = ET.parse(xml_path)
        root = tree.getroot()

        image_name = xml_file.replace(
            ".xml",
            ".jpg"
        )

        image_path = os.path.join(
            img_dir,
            image_name
        )

        for obj in root.findall("object"):

            bbox = obj.find("bndbox")

            xmin = int(
                bbox.find("xmin").text
            )

            ymin = int(
                bbox.find("ymin").text
            )

            xmax = int(
                bbox.find("xmax").text
            )

            ymax = int(
                bbox.find("ymax").text
            )

            records.append([
                image_path,
                image_name,
                cls,
                CLASS_TO_ID[cls],
                xmin,
                ymin,
                xmax,
                ymax
            ])

df = pd.DataFrame(

    records,

    columns=[

        "image_path",
        "image_name",

        "class_name",
        "class_id",

        "xmin",
        "ymin",
        "xmax",
        "ymax"
    ]
)

print(df.head())

print("\nShape:")
print(df.shape)

print("\nClass Distribution:")
print(df["class_name"].value_counts())

                                          image_path              image_name  \
0  PCB-DATASET-master\images\Missing_hole\01_miss...  01_missing_hole_01.jpg   
1  PCB-DATASET-master\images\Missing_hole\01_miss...  01_missing_hole_01.jpg   
2  PCB-DATASET-master\images\Missing_hole\01_miss...  01_missing_hole_01.jpg   
3  PCB-DATASET-master\images\Missing_hole\01_miss...  01_missing_hole_02.jpg   
4  PCB-DATASET-master\images\Missing_hole\01_miss...  01_missing_hole_02.jpg   

     class_name  class_id  xmin  ymin  xmax  ymax  
0  Missing_hole         0  2459  1274  2530  1329  
1  Missing_hole         0  1613   334  1679   396  
2  Missing_hole         0  1726   794  1797   854  
3  Missing_hole         0  2584   232  2650   298  
4  Missing_hole         0  2366   803  2406   860  

Shape:
(2953, 8)

Class Distribution:
class_name
Spurious_copper    503
Missing_hole       497
Mouse_bite         492
Short              491
Spur               488
Open_circuit       482
Name: count, dtype:

In [12]:
# Print basic dataset statistics and defects-per-image summary values.

print(
    "Unique Images:",
    df["image_name"].nunique()
)

print()

print(
    df.groupby(
        "image_name"
    ).size().describe()
)

print()

print(
    "Maximum defects in one image:",
    df.groupby(
        "image_name"
    ).size().max()
)

print(
    "Minimum defects in one image:",
    df.groupby(
        "image_name"
    ).size().min()
)

Unique Images: 693

count    693.000000
mean       4.261183
std        1.035306
min        1.000000
25%        3.000000
50%        5.000000
75%        5.000000
max        6.000000
dtype: float64

Maximum defects in one image: 6
Minimum defects in one image: 1


In [13]:
# Extract the unique image names used for image-level splitting.

unique_images = df["image_name"].unique()

print(
    "Total Images:",
    len(unique_images)
)

Total Images: 693


In [14]:
# Split images into training and temporary holdout sets.

train_images, temp_images = train_test_split(

    unique_images,

    test_size=0.30,

    random_state=SEED,

    shuffle=True
)

In [15]:
# Split the temporary holdout images into validation and test sets.

val_images, test_images = train_test_split(

    temp_images,

    test_size=1/3,

    random_state=SEED,

    shuffle=True
)

In [16]:
# Create annotation DataFrames for the train, validation, and test splits.

train_df = df[
    df["image_name"].isin(
        train_images
    )
].reset_index(drop=True)

val_df = df[
    df["image_name"].isin(
        val_images
    )
].reset_index(drop=True)

test_df = df[
    df["image_name"].isin(
        test_images
    )
].reset_index(drop=True)

In [17]:
# Print image and object counts for each data split.

print(
    "Train Images:",
    len(train_images)
)

print(
    "Val Images:",
    len(val_images)
)

print(
    "Test Images:",
    len(test_images)
)

print()

print(
    "Train Objects:",
    len(train_df)
)

print(
    "Val Objects:",
    len(val_df)
)

print(
    "Test Objects:",
    len(test_df)
)

Train Images: 485
Val Images: 138
Test Images: 70

Train Objects: 2050
Val Objects: 600
Test Objects: 303


In [18]:
# Check that train, validation, and test image sets do not overlap.

print(
    "Train-Val overlap:",
    len(
        set(train_images)
        &
        set(val_images)
    )
)

print(
    "Train-Test overlap:",
    len(
        set(train_images)
        &
        set(test_images)
    )
)

print(
    "Val-Test overlap:",
    len(
        set(val_images)
        &
        set(test_images)
    )
)

Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0


In [19]:
# Show the training class distribution.

print(
    train_df["class_name"]
    .value_counts()
)

class_name
Short              375
Spur               350
Spurious_copper    348
Open_circuit       341
Mouse_bite         340
Missing_hole       296
Name: count, dtype: int64


In [20]:
# Show the validation class distribution.

print(
    val_df["class_name"]
    .value_counts()
)

class_name
Missing_hole       135
Mouse_bite         116
Spurious_copper    109
Open_circuit        90
Spur                82
Short               68
Name: count, dtype: int64


In [21]:
# Show the test class distribution.

print(
    test_df["class_name"]
    .value_counts()
)

class_name
Missing_hole       66
Spur               56
Open_circuit       51
Short              48
Spurious_copper    46
Mouse_bite         36
Name: count, dtype: int64


In [22]:
# Define training augmentations and bounding-box handling.

train_transform = A.Compose(

    [

        A.Resize(
            800,
            800
        ),

        A.HorizontalFlip(
            p=0.5
        ),

        A.VerticalFlip(
            p=0.5
        ),

        A.RandomBrightnessContrast(
            brightness_limit=0.03,
            contrast_limit=0.03,
            p=0.3
        )

    ],

    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["labels"]
    )

)

In [23]:
# Define validation and test preprocessing.

val_transform = A.Compose(

    [

        A.Resize(
            800,
            800
        )

    ],

    bbox_params=A.BboxParams(
        format="pascal_voc",
        label_fields=["labels"]
    )

)

In [24]:
# Define the PyTorch dataset that loads images, boxes, labels, and transforms.

class PCBDataset(Dataset):

    def __init__(
        self,
        dataframe,
        transform=None
    ):

        self.df = dataframe

        self.transform = transform

        self.images = sorted(
            dataframe["image_name"]
            .unique()
        )

    def __len__(self):

        return len(
            self.images
        )

    def __getitem__(
        self,
        idx
    ):

        image_name = self.images[idx]

        records = self.df[
            self.df["image_name"]
            ==
            image_name
        ]

        image_path = records[
            "image_path"
        ].iloc[0]

        image = cv2.imread(
            image_path
        )

        image = cv2.cvtColor(
            image,
            cv2.COLOR_BGR2RGB
        )

        boxes = records[
            [
                "xmin",
                "ymin",
                "xmax",
                "ymax"
            ]
        ].values

        labels = records[
            "class_id"
        ].values

        if self.transform:

            transformed = self.transform(

                image=image,

                bboxes=boxes,

                labels=labels

            )

            image = transformed[
                "image"
            ]

            boxes = transformed[
                "bboxes"
            ]

            labels = transformed[
                "labels"
            ]

        image = image.astype(
            np.float32
        ) / 255.0

        image = torch.tensor(
            image
        ).permute(
            2,
            0,
            1
        )

        target = {

            "boxes": torch.tensor(
                boxes,
                dtype=torch.float32
            ),

            "labels": torch.tensor(
                labels,
                dtype=torch.long
            )

        }

        return image, target

In [25]:
# Instantiate train, validation, and test datasets.

train_dataset = PCBDataset(
    train_df,
    transform=train_transform
)

val_dataset = PCBDataset(
    val_df,
    transform=val_transform
)

test_dataset = PCBDataset(
    test_df,
    transform=val_transform
)

In [26]:
# Print dataset sizes for each split.

print(
    len(train_dataset)
)

print(
    len(val_dataset)
)

print(
    len(test_dataset)
)

485
138
70


In [27]:
# Inspect one dataset sample and its target tensor shapes.

image, target = train_dataset[0]

print(image.shape)

print(
    target["boxes"].shape
)

print(
    target["labels"].shape
)

torch.Size([3, 800, 800])
torch.Size([3, 4])
torch.Size([3])


In [28]:
# Define the custom collate function for variable numbers of boxes per image.

def collate_fn(batch):

    images = []
    targets = []

    for image, target in batch:

        images.append(image)
        targets.append(target)

    images = torch.stack(images)

    return images, targets

In [29]:
# Set the DataLoader batch size.

BATCH_SIZE = 8

In [30]:
# Build DataLoaders for training, validation, and testing.

train_loader = DataLoader(

    train_dataset,

    batch_size=BATCH_SIZE,

    shuffle=True,

    num_workers=2,

    collate_fn=collate_fn

)

val_loader = DataLoader(

    val_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    collate_fn=collate_fn

)

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=2,

    collate_fn=collate_fn

)

In [ ]:
# Inspect the shape and target structure of one training batch.

images, targets = next(
    iter(train_loader)
)

print(images.shape)

print(targets[0].keys())

print(targets[0]["boxes"].shape)

print(targets[0]["labels"].shape)

In [ ]:
# Visualize random augmented training samples with their bounding boxes.

for _ in range(10):

    idx = random.randint(
        0,
        len(train_dataset)-1
    )

    image, target = train_dataset[idx]

    img = image.permute(
        1,
        2,
        0
    ).numpy()

    fig, ax = plt.subplots(
        figsize=(8,8)
    )

    ax.imshow(img)

    for box, label in zip(

        target["boxes"],
        target["labels"]

    ):

        xmin, ymin, xmax, ymax = box.numpy()

        rect = patches.Rectangle(

            (xmin, ymin),

            xmax - xmin,

            ymax - ymin,

            linewidth=2,

            edgecolor="red",

            facecolor="none"

        )

        ax.add_patch(rect)

        ax.text(

            xmin,

            ymin - 3,

            CLASSES[
                int(label)
            ],

            color="white",

            fontsize=8,

            bbox=dict(
                facecolor="red",
                alpha=0.8
            )

        )

    ax.axis("off")

    plt.show()

## Phase 2: Encode Bounding Boxes into Grid Targets


In [ ]:
# Define the image/grid geometry used by the detector target format.

IMAGE_SIZE = 800

GRID_SIZE = 100

STRIDE = IMAGE_SIZE // GRID_SIZE

NUM_CLASSES = len(CLASSES)

print("Stride:", STRIDE)

In [ ]:
# Encode boxes and labels into objectness, class, offset, width, and height maps.

def encode_targets(
    boxes,
    labels
):

    target = np.zeros(
        (
            11,
            GRID_SIZE,
            GRID_SIZE
        ),
        dtype=np.float32
    )

    for box, label in zip(
        boxes,
        labels
    ):

        xmin, ymin, xmax, ymax = box

        cx = (xmin + xmax) / 2
        cy = (ymin + ymax) / 2

        w = xmax - xmin
        h = ymax - ymin

        grid_x = int(cx / STRIDE)
        grid_y = int(cy / STRIDE)

        if (
            grid_x >= GRID_SIZE
            or
            grid_y >= GRID_SIZE
        ):
            continue

        # Objectness

        target[
            0,
            grid_y,
            grid_x
        ] = 1.0

        # One-hot class

        target[
            1 + int(label),
            grid_y,
            grid_x
        ] = 1.0

        # Offsets inside cell

        target[
            7,
            grid_y,
            grid_x
        ] = (
            cx / STRIDE
            -
            grid_x
        )

        target[
            8,
            grid_y,
            grid_x
        ] = (
            cy / STRIDE
            -
            grid_y
        )

        # Normalized width / height

        target[
            9,
            grid_y,
            grid_x
        ] = w / IMAGE_SIZE

        target[
            10,
            grid_y,
            grid_x
        ] = h / IMAGE_SIZE

    return target

In [ ]:
# Encode one sample and inspect the encoded target shape.

image, target = train_dataset[0]

encoded = encode_targets(
    target["boxes"].numpy(),
    target["labels"].numpy()
)

print(encoded.shape)

In [ ]:
# Compare the number of positive grid cells with the number of ground-truth boxes.

obj_map = encoded[0]

print(
    "Positive Cells:",
    int(obj_map.sum())
)

print(
    "GT Boxes:",
    len(target["boxes"])
)

In [ ]:
# Visualize the objectness map for one encoded sample.

plt.figure(figsize=(6,6))

plt.imshow(
    encoded[0],
    cmap="hot"
)

plt.title(
    "Objectness Map"
)

plt.colorbar()

plt.show()

In [ ]:
# Count active class channels in the encoded sample.

for cls in range(NUM_CLASSES):

    count = encoded[
        1 + cls
    ].sum()

    print(
        CLASSES[cls],
        int(count)
    )

## Phase 3: Build Batch Targets and Define the Detector Network


In [ ]:
# Convert a list of per-image targets into a batch target tensor.

def build_batch_targets(
    targets
):

    batch = []

    for target in targets:

        encoded = encode_targets(

            target["boxes"].numpy(),

            target["labels"].numpy()

        )

        batch.append(encoded)

    batch = np.stack(
        batch,
        axis=0
    )

    batch = torch.tensor(
        batch,
        dtype=torch.float32
    )

    return batch

In [ ]:
# Build one batch of encoded targets and inspect its shape.

images, targets = next(
    iter(train_loader)
)

batch_targets = build_batch_targets(
    targets
)

print(images.shape)

print(batch_targets.shape)

print(
    "Positive Cells:",
    batch_targets[:,0].sum()
)

In [ ]:
# Define a reusable convolution, batch-normalization, and ReLU block.

class ConvBlock(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        stride=1
    ):

        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                stride=stride,
                padding=1,
                bias=False
            ),

            nn.BatchNorm2d(
                out_channels
            ),

            nn.ReLU(inplace=True)

        )

    def forward(
        self,
        x
    ):

        return self.block(x)

In [ ]:
# Define the CNN detector that predicts 11 channels over a 100x100 grid.

class PCBDetectorV3(nn.Module):

    def __init__(self):

        super().__init__()

        # 800 ﷿﷿﷿﷿﷿﷿﷿﷿﷿ 400

        self.stage1 = nn.Sequential(

            ConvBlock(3,32),

            ConvBlock(32,32),

            ConvBlock(
                32,
                64,
                stride=2
            )

        )

        # 400 ﷿﷿﷿﷿﷿﷿﷿﷿﷿ 200

        self.stage2 = nn.Sequential(

            ConvBlock(64,64),

            ConvBlock(64,64),

            ConvBlock(
                64,
                128,
                stride=2
            )

        )

        # 200 ﷿﷿﷿﷿﷿﷿﷿﷿﷿ 100

        self.stage3 = nn.Sequential(

            ConvBlock(128,128),

            ConvBlock(128,128),

            ConvBlock(
                128,
                256,
                stride=2
            )

        )

        self.head = nn.Sequential(

            ConvBlock(
                256,
                256
            ),

            ConvBlock(
                256,
                128
            ),

            nn.Conv2d(
                128,
                11,
                kernel_size=1
            )

        )

    def forward(
        self,
        x
    ):

        x = self.stage1(x)

        x = self.stage2(x)

        x = self.stage3(x)

        x = self.head(x)

        return x

In [ ]:
# Select the compute device and instantiate the detector.

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

model = PCBDetectorV3().to(
    DEVICE
)

In [ ]:
# Count the trainable model parameters.

num_params = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)

print(
    f"Parameters: {num_params:,}"
)

In [ ]:
# Run one batch through the model and inspect the output shape.

images, targets = next(
    iter(train_loader)
)

images = images.to(
    DEVICE
)

with torch.no_grad():

    outputs = model(
        images
    )

print(
    outputs.shape
)

In [ ]:
# Print quick model-output and target-count sanity checks.

print(outputs.shape)

print(
    f"Parameters: {num_params:,}"
)

print(
    "Positive Cells:",
    batch_targets[:,0].sum()
)

## Phase 4: Define the Detection Loss


In [ ]:
# Define the objectness, classification, and box-regression loss.

class DetectionLoss(nn.Module):

    def __init__(self):

        super().__init__()

        self.obj_loss_fn = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        [20.0],
        device=DEVICE
    )
)

        self.box_loss_fn = nn.SmoothL1Loss()

        self.cls_loss_fn = nn.CrossEntropyLoss()

    def forward(
        self,
        preds,
        targets
    ):

        # ------------------
        # Objectness
        # ------------------

        pred_obj = preds[:,0]

        target_obj = targets[:,0]

        obj_loss = self.obj_loss_fn(
            pred_obj,
            target_obj
        )

        # ------------------
        # Positive cells
        # ------------------

        pos_mask = (
            target_obj > 0
        )

        # no positives safety

        if pos_mask.sum() == 0:

            return {

                "total_loss": obj_loss,

                "obj_loss": obj_loss,

                "cls_loss": torch.tensor(
                    0.0,
                    device=preds.device
                ),

                "box_loss": torch.tensor(
                    0.0,
                    device=preds.device
                )

            }

        # ------------------
        # Classification
        # ------------------

        pred_cls = preds[:,1:7]

        target_cls = targets[:,1:7]

        target_class_ids = (
            target_cls.argmax(
                dim=1
            )
        )

        pred_cls_pos = pred_cls.permute(
            0,
            2,
            3,
            1
        )[pos_mask]

        target_cls_pos = (
            target_class_ids[
                pos_mask
            ]
        )

        cls_loss = self.cls_loss_fn(

            pred_cls_pos,

            target_cls_pos

        )

        # ------------------
        # Box Regression
        # ------------------

        pred_box = preds[:,7:11]

        target_box = targets[:,7:11]

        pred_box_pos = pred_box.permute(
            0,
            2,
            3,
            1
        )[pos_mask]

        target_box_pos = target_box.permute(
            0,
            2,
            3,
            1
        )[pos_mask]

        box_loss = self.box_loss_fn(

            pred_box_pos,

            target_box_pos

        )

        total_loss = (

            obj_loss

            + cls_loss

            + 5.0 * box_loss

        )

        return {

            "total_loss": total_loss,

            "obj_loss": obj_loss,

            "cls_loss": cls_loss,

            "box_loss": box_loss

        }

In [ ]:
# Instantiate the detection loss.

criterion = DetectionLoss()

In [ ]:
# Compute loss components on one training batch as a sanity check.

images, targets = next(
    iter(train_loader)
)

images = images.to(
    DEVICE
)

batch_targets = build_batch_targets(
    targets
).to(DEVICE)

preds = model(
    images
)

losses = criterion(
    preds,
    batch_targets
)

for k,v in losses.items():

    print(
        k,
        float(v)
    )

## Phase 5: Train, Save, Decode, and Inspect Predictions


In [ ]:
# Configure the AdamW optimizer.

optimizer = torch.optim.AdamW(

    model.parameters(),

    lr=1e-3,

    weight_decay=1e-4

)

In [ ]:
# Configure the validation-loss learning-rate scheduler.

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=0.5,

    patience=3
)

In [ ]:
# Set training length, early-stopping patience, and best-loss tracking.

EPOCHS = 100

PATIENCE = 10

best_val_loss = float("inf")

patience_counter = 0

In [ ]:
# Define one full training epoch and return averaged loss components.

def train_one_epoch():

    model.train()

    total_loss_sum = 0
    obj_loss_sum = 0
    cls_loss_sum = 0
    box_loss_sum = 0

    for images, targets in train_loader:

        images = images.to(DEVICE)

        batch_targets = build_batch_targets(
            targets
        ).to(DEVICE)

        optimizer.zero_grad()

        preds = model(images)

        losses = criterion(
            preds,
            batch_targets
        )

        losses["total_loss"].backward()

        optimizer.step()

        total_loss_sum += losses["total_loss"].item()

        obj_loss_sum += losses["obj_loss"].item()

        cls_loss_sum += losses["cls_loss"].item()

        box_loss_sum += losses["box_loss"].item()

    n = len(train_loader)

    return {

        "total": total_loss_sum / n,

        "obj": obj_loss_sum / n,

        "cls": cls_loss_sum / n,

        "box": box_loss_sum / n

    }

In [ ]:
# Define one validation epoch without gradient updates.

@torch.no_grad()
def validate():

    model.eval()

    total_loss_sum = 0
    obj_loss_sum = 0
    cls_loss_sum = 0
    box_loss_sum = 0

    for images, targets in val_loader:

        images = images.to(DEVICE)

        batch_targets = build_batch_targets(
            targets
        ).to(DEVICE)

        preds = model(images)

        losses = criterion(
            preds,
            batch_targets
        )

        total_loss_sum += losses["total_loss"].item()

        obj_loss_sum += losses["obj_loss"].item()

        cls_loss_sum += losses["cls_loss"].item()

        box_loss_sum += losses["box_loss"].item()

    n = len(val_loader)

    return {

        "total": total_loss_sum / n,

        "obj": obj_loss_sum / n,

        "cls": cls_loss_sum / n,

        "box": box_loss_sum / n

    }

In [ ]:
# Train the model, save the best checkpoint, and stop early when validation stalls.

for epoch in range(EPOCHS):

    train_metrics = train_one_epoch()

    val_metrics = validate()

    scheduler.step(
        val_metrics["total"]
    )

    current_lr = optimizer.param_groups[0]["lr"]

    print()

    print(
        f"Epoch [{epoch+1}/{EPOCHS}]"
    )

    print(
        f"LR: {current_lr:.6f}"
    )

    print(
        f"Train Total: {train_metrics['total']:.4f}"
    )

    print(
        f"Val Total:   {val_metrics['total']:.4f}"
    )

    print(
        f"Train Obj: {train_metrics['obj']:.4f}"
    )

    print(
        f"Val Obj:   {val_metrics['obj']:.4f}"
    )

    print(
        f"Train Cls: {train_metrics['cls']:.4f}"
    )

    print(
        f"Val Cls:   {val_metrics['cls']:.4f}"
    )

    print(
        f"Train Box: {train_metrics['box']:.4f}"
    )

    print(
        f"Val Box:   {val_metrics['box']:.4f}"
    )

    # -------------------
    # Save best
    # -------------------

    if val_metrics["total"] < best_val_loss:

        best_val_loss = val_metrics["total"]

        patience_counter = 0

        torch.save(

            model.state_dict(),

            "best_detector_v3.pth"

        )

        print(
            "﷿﷿﷿﷿﷿﷿﷿﷿﷿ Best model saved"
        )

    else:

        patience_counter += 1

    # -------------------
    # Early stop
    # -------------------

    if patience_counter >= PATIENCE:

        print()

        print(
            "Early stopping triggered."
        )

        break

In [ ]:
# Load the saved best detector checkpoint.

model.load_state_dict(
    torch.load(
        "best_detector_v3.pth",
        map_location=DEVICE
    )
)

model.eval()

In [ ]:
# Decode model grid outputs into boxes, labels, and confidence scores.

def decode_predictions(
    outputs,
    score_threshold=0.30
):

    outputs = outputs.detach().cpu()

    batch_predictions = []

    B = outputs.shape[0]

    for b in range(B):

        pred = outputs[b]

        # ------------------
        # Objectness
        # ------------------

        objectness = torch.sigmoid(
            pred[0]
        )

        # ------------------
        # Classes
        # ------------------

        class_probs = torch.softmax(
            pred[1:7],
            dim=0
        )

        boxes = []
        labels = []
        scores = []

        ys, xs = torch.where(
            objectness > score_threshold
        )

        for y, x in zip(ys, xs):

            score = objectness[
                y,
                x
            ].item()

            cls = torch.argmax(
                class_probs[:, y, x]
            ).item()

            cx_offset = pred[
                7,
                y,
                x
            ].item()

            cy_offset = pred[
                8,
                y,
                x
            ].item()

            width = pred[
                9,
                y,
                x
            ].item()

            height = pred[
                10,
                y,
                x
            ].item()

            # ------------------
            # Recover center
            # ------------------

            cx = (
                x + cx_offset
            ) * STRIDE

            cy = (
                y + cy_offset
            ) * STRIDE

            width *= IMAGE_SIZE
            height *= IMAGE_SIZE

            xmin = cx - width/2
            ymin = cy - height/2

            xmax = cx + width/2
            ymax = cy + height/2

            boxes.append([
                xmin,
                ymin,
                xmax,
                ymax
            ])

            labels.append(cls)

            scores.append(score)

        batch_predictions.append({

            "boxes": np.array(boxes),

            "labels": np.array(labels),

            "scores": np.array(scores)

        })

    return batch_predictions

In [ ]:
# Reload the best checkpoint for prediction inspection.

model.load_state_dict(

    torch.load(

        "best_detector_v3.pth",

        map_location=DEVICE

    )

)

model.eval()

print(
    "Best model loaded."
)

In [ ]:
# Run inference on one validation batch and inspect decoded prediction structure.

images, targets = next(
    iter(val_loader)
)

images = images.to(
    DEVICE
)

with torch.no_grad():

    outputs = model(
        images
    )

preds = decode_predictions(
    outputs,
    score_threshold=0.50
)

print(
    len(preds)
)

print(
    preds[0].keys()
)

print(
    len(preds[0]["boxes"])
)

In [ ]:
# Visualize random validation predictions against ground-truth boxes.

for _ in range(10):

    idx = random.randint(
        0,
        len(val_dataset)-1
    )

    image, target = val_dataset[idx]

    with torch.no_grad():

        outputs = model(

            image
            .unsqueeze(0)
            .to(DEVICE)

        )

    preds = decode_predictions(

        outputs,

        score_threshold=0.60

    )[0]

    img = image.permute(
        1,
        2,
        0
    ).numpy()

    fig, ax = plt.subplots(
        figsize=(10,10)
    )

    ax.imshow(img)

    ax.set_title(

        f"GT={len(target['boxes'])} | "
        f"Pred={len(preds['boxes'])}"

    )

    # -------------------
    # Ground Truth (GREEN)
    # -------------------

    for box, label in zip(

        target["boxes"],

        target["labels"]

    ):

        xmin, ymin, xmax, ymax = box.numpy()

        rect = patches.Rectangle(

            (xmin, ymin),

            xmax - xmin,

            ymax - ymin,

            linewidth=2,

            edgecolor="lime",

            facecolor="none"

        )

        ax.add_patch(rect)

        ax.text(

            xmin,

            ymin - 2,

            CLASSES[
                int(label)
            ],

            color="black",

            fontsize=8,

            bbox=dict(
                facecolor="lime",
                alpha=0.8
            )

        )

    # -------------------
    # Predictions (RED)
    # -------------------

    for box, label, score in zip(

        preds["boxes"],

        preds["labels"],

        preds["scores"]

    ):

        xmin, ymin, xmax, ymax = box

        rect = patches.Rectangle(

            (xmin, ymin),

            xmax - xmin,

            ymax - ymin,

            linewidth=2,

            edgecolor="red",

            facecolor="none"

        )

        ax.add_patch(rect)

        ax.text(

            xmin,

            ymax + 5,

            f"{CLASSES[int(label)]} {score:.2f}",

            color="white",

            fontsize=7,

            bbox=dict(
                facecolor="red",
                alpha=0.8
            )

        )

    # -------------------
    # Console Summary
    # -------------------

    print("\n" + "="*60)

    print(
        f"GT Defects: {len(target['boxes'])}"
    )

    print(
        f"Predictions: {len(preds['boxes'])}"
    )

    print("\nGround Truth Classes:")

    gt_counter = {}

    for label in target["labels"]:

        cls_name = CLASSES[int(label)]

        gt_counter[cls_name] = (
            gt_counter.get(cls_name, 0) + 1
        )

    for cls_name, count in gt_counter.items():

        print(
            f"  - {cls_name:<20} x{count}"
        )

    print("\nPredicted Classes:")

    if len(preds["labels"]) == 0:

        print("  No predictions")

    else:

        pred_counter = {}

        for cls in preds["labels"]:

            cls_name = CLASSES[int(cls)]

            pred_counter[cls_name] = (
                pred_counter.get(cls_name, 0) + 1
            )

        for cls_name, count in pred_counter.items():

            print(
                f"  - {cls_name:<20} x{count}"
            )

        print("\nPrediction Details:")

        for i, (cls, score) in enumerate(

            zip(
                preds["labels"],
                preds["scores"]
            )

        ):

            print(

                f"[{i+1:02d}] "

                f"{CLASSES[int(cls)]:<20} "

                f"Confidence = {score:.3f}"

            )

    print("="*60)

    ax.axis("off")

    plt.show()

In [ ]:
# Inspect objectness score percentiles across validation outputs.

all_scores = []

model.eval()

with torch.no_grad():

    for images, _ in val_loader:

        images = images.to(DEVICE)

        outputs = model(images)

        obj_scores = torch.sigmoid(
            outputs[:,0]
        )

        positive_scores = obj_scores[
            obj_scores > 0.01
        ]

        all_scores.extend(
            positive_scores
            .cpu()
            .numpy()
            .tolist()
        )

print("Count:", len(all_scores))

print(
    np.percentile(
        all_scores,
        [50,75,90,95,99]
    )
)

In [ ]:
# Inspect confidence percentiles after high-threshold decoding.

all_scores = []

for images, _ in val_loader:

    images = images.to(DEVICE)

    with torch.no_grad():

        outputs = model(images)

    preds = decode_predictions(
        outputs,
        score_threshold=0.90
    )

    for pred in preds:

        all_scores.extend(
            pred["scores"]
        )

print(
    np.percentile(
        all_scores,
        [50,75,90,95,99]
    )
)

In [ ]:
# Summarize how many predictions are produced per validation image.

pred_counts = []

model.eval()

with torch.no_grad():

    for images, _ in val_loader:

        images = images.to(DEVICE)

        outputs = model(images)

        preds = decode_predictions(
            outputs,
            score_threshold=0.50
        )

        for pred in preds:

            pred_counts.append(
                len(pred["boxes"])
            )

print("Mean predictions/image:",
      np.mean(pred_counts))

print("Median predictions/image:",
      np.median(pred_counts))

print("Min:",
      np.min(pred_counts))

print("Max:",
      np.max(pred_counts))